# Image FetchPush conedir - OFFLINE ORIGINAL-STYLE qualification (fixed alpha=0 + BC 0.05 + twin)

```
actor_loss = 0.05 * BC_NLL(data_action | state, future_goal)
           + 0.95 * (-min_Q(state, policy_action, future_goal))
```
Fixed alpha=0, bc_coef=0.05, twin_q (min), random_goals=0, frozen offline .npz.
All eval distances are PHYSICAL object-goal coords, never image-L2.
Requires code commit **a6d8373** or later (set COMMIT in cell 2).

In [ ]:
# 1. Dependencies WITHOUT disturbing the preinstalled Colab GPU JAX (proven-safe recipe).
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["MUJOCO_GL"] = "egl"
import jax, jaxlib, numpy, subprocess, sys
hold = [f"jax=={jax.__version__}", f"jaxlib=={jaxlib.__version__}", f"numpy=={numpy.__version__}"]
def pip(*a): subprocess.run([sys.executable,"-m","pip","install","-q",*a], check=True)
print("Colab JAX", jax.__version__, "| devices:", jax.devices())
pip("--no-deps","dm-haiku","optax","chex")
pip("jmp","tabulate","toolz","etils","tensorboardX","gymnasium-robotics","mujoco","imageio-ffmpeg",*hold)
print("post-install JAX", jax.__version__, "| devices:", jax.devices())

In [ ]:
# 2. Clone the fork and checkout the commit with the driver + instrumentation + fix.
import os
os.chdir("/content")
if not os.path.exists("/content/contrastive_rl"):
    !git clone https://github.com/tingrui-huang/contrastive_rl.git
%cd /content/contrastive_rl
COMMIT = "a6d8373"   # driver + physical-eval instrumentation + final-save None-guard fix
!git fetch -q origin && git checkout -q $COMMIT
!git log -1 --oneline
assert os.path.exists("scripts/qualify_image_offline_original_style.py"), "missing driver -- set COMMIT to a6d8373 or later."
assert os.path.exists("scripts/collect_push_dataset.py"), "missing dataset collector."

In [ ]:
# 3. Mount Google Drive.
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 4. Paths. Dataset collected ONCE (cell 6), then reused. New run dir.
import os, json, shutil, numpy as np
ENV  = "fetch_push_image_conedir"
SEED = 0
SMOKE = True     # True = tiny smoke; set False for the real 100k qualification
BASE = "/content/drive/MyDrive/easypush_image_qual"
DATA = f"{BASE}/data/push_image_conedir_noisy_oracle_s{SEED}" + ("_smoke" if SMOKE else "") + ".npz"
RUN_DIR = f"{BASE}/fetch_push_image_conedir_off_bc005_alpha0_twin_s{SEED}" + ("_smoke" if SMOKE else "")
OUT = RUN_DIR + "/analysis"     # on DRIVE so verdict/CSVs survive disconnects
STEPS = 2000 if SMOKE else 100000
N_EPISODES = 50 if SMOKE else 1000
_finished = os.path.exists(os.path.join(RUN_DIR, "final.pkl"))
if SMOKE and _finished:
    shutil.rmtree(RUN_DIR); print("cleared previous smoke run dir:", RUN_DIR)
elif _finished:
    print("NOTE:", RUN_DIR, "already has a FINISHED real run. Running the TRAIN cell",
          "would retrain+overwrite it -- to just regenerate results use the POST-HOC cell.")
print("DATA   :", DATA, "(exists)" if os.path.exists(DATA) else "(will collect)")
print("RUN_DIR:", RUN_DIR)

In [ ]:
# 5. Environment/render audit -- STOP if it fails.
!python -m scripts.smoke_image_conedir
audit = json.load(open("artifacts/push_image_probe/smoke_image_conedir.json"))
assert audit["verdict"] == "PASS", "image-conedir env audit FAILED -- stopping."
print("env audit PASS (oracle=%.2f random=%.2f)" % (audit["gate4_oracle"]["oracle_success"], audit["gate4_oracle"]["random_success"]))

In [ ]:
# 6. Collect the FROZEN dataset ONCE (renders the env; NOT a download). Reused if present.
os.makedirs(os.path.dirname(DATA), exist_ok=True)
if os.path.exists(DATA):
    meta = json.loads(str(np.load(DATA, allow_pickle=True)["meta"])); print("reusing dataset:", DATA)
else:
    from scripts.collect_push_dataset import collect
    print("collecting", N_EPISODES, "episodes ->", DATA)
    meta = collect(ENV, episodes=N_EPISODES, noise=0.3, random_frac=0.2, seed=SEED, out=DATA)
print(json.dumps(meta, indent=2))
assert meta["behavior_success_oracle"] and meta["behavior_success_oracle"] >= 0.9, "demonstrator broken"
print("dataset ready, behavior_success =", meta["behavior_success"])

In [ ]:
# 7. Launch TensorBoard on the run dir (physical success/dist + actor/critic diagnostics,
#    updates live as the driver in cell 8 writes RUN_DIR/tb).
tb_logdir = RUN_DIR + "/tb"
%load_ext tensorboard
%tensorboard --logdir $tb_logdir

In [ ]:
# 8. Run the qualification driver (preflight -> offline train -> physical fixed-eval ->
#    checkpoint diagnostics -> rollout GIFs -> verdict). Real traceback printed on error.
cmd = ["python","-m","scripts.qualify_image_offline_original_style",
       "--dataset", DATA, "--run_dir", RUN_DIR, "--out", OUT,
       "--seed", str(SEED), "--steps", str(STEPS)]
if SMOKE: cmd.append("--smoke")
print(" ".join(cmd))
import subprocess
r = subprocess.run(cmd)
if r.returncode != 0:
    raise SystemExit("driver failed (exit %d); see traceback above" % r.returncode)

### Recovery: regenerate results without retraining

If the runtime disconnected AFTER training finished (checkpoints are on Drive) but
you lost the verdict/CSVs, run cells 1-4 then the cell below. `--posthoc-only`
skips preflight+training and rebuilds `fixed_eval.csv` / `checkpoint_diagnostics.csv`
/ `summary.json` + GIFs from the saved checkpoints in a few minutes. (Skip this cell
for a normal run.)

In [ ]:
# (optional) POST-HOC ONLY -- reuse Drive checkpoints, no retraining.
r = __import__("subprocess").run(["python","-m","scripts.qualify_image_offline_original_style",
     "--dataset", DATA, "--run_dir", RUN_DIR, "--out", OUT, "--seed", str(SEED),
     "--posthoc-only"] + (["--smoke"] if SMOKE else []))
print("exit", r.returncode)

In [ ]:
# 9. Show results: preflight, verdict, CSVs, and ALL rollout GIFs (any step, glob-matched).
import json, os, glob
from IPython.display import Image, display
if os.path.exists(f"{OUT}/preflight_audit.json"):
    print("=== PREFLIGHT ===")
    print(json.dumps(json.load(open(f"{OUT}/preflight_audit.json"))["checks"], indent=2))
summ = json.load(open(f"{OUT}/summary.json"))
print()
print("=== VERDICT:", summ["qualification_verdict"], "===")
print(json.dumps(summ["final_checkpoint_physical"], indent=2))
print()
print("=== fixed_eval.csv ===")
print(open(f"{OUT}/fixed_eval.csv").read())
print("=== checkpoint_diagnostics.csv ===")
print(open(f"{OUT}/checkpoint_diagnostics.csv").read())
gifs = sorted(glob.glob(os.path.join(RUN_DIR, "rollout_*.gif")))
print("=== rollout GIFs:", [os.path.basename(g) for g in gifs], "===")
for g in gifs:
    print(os.path.basename(g)); display(Image(g, width=256))